# 03 · Ablation kiến trúc

Hai thí nghiệm bắt buộc còn lại, đều dùng hàm mất mát tốt nhất tìm được ở
notebook 02:

1. **Cách tăng mẫu**: `transpose` / `bilinear` / `unpool`
2. **Skip connection**: `full` / `half` / `none`

Cấu hình gốc đã chạy rồi nên chỉ cần thêm 4 lượt mới.

Người phụ trách: **SV A**.

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

In [ ]:
BEST_LOSS = "bce_dice"   # cập nhật theo kết quả notebook 02
base = Config(loss_name=BEST_LOSS, epochs=120, seed=42)

up_configs = [base.replace(up_mode=m, skip_mode="full", tag="ablation_up")
              for m in UP_MODES]
skip_configs = [base.replace(up_mode="transpose", skip_mode=m, tag="ablation_skip")
                for m in SKIP_MODES]

for c in up_configs + skip_configs:
    print(f"{c.run_id:50s}")

Số tham số **không bằng nhau** giữa ba cách tăng mẫu và ba mức skip.
Nêu rõ điểm này trong báo cáo như một hạn chế của thiết kế thí nghiệm — giám
khảo hay hỏi đúng chỗ này.

In [ ]:
for c in up_configs + skip_configs:
    m = build_model(c)
    print(f"{c.up_mode:10s} {c.skip_mode:5s} {count_parameters(m):>12,} tham số")
    del m

In [ ]:
results = run_sweep(up_configs + skip_configs)

## Hai bảng ablation

In [ ]:
import pandas as pd

df = RunLogger(base.log_csv).to_dataframe()
num = ["test_dice", "test_iou", "test_precision", "test_recall", "n_params"]
df[num] = df[num].astype(float)

print("=== Cách tăng mẫu ===")
print(df[df.tag == "ablation_up"][["up_mode"] + num].round(4).to_markdown(index=False))
print("\n=== Skip connection ===")
print(df[df.tag == "ablation_skip"][["skip_mode"] + num].round(4).to_markdown(index=False))

## Soi checkerboard artifact

Phải phóng to sát biên polyp mới thấy. Không zoom thì phần bình luận trong
báo cáo sẽ thành nói suông.

In [ ]:
from src.viz import zoom_on_border, predict_batch
import numpy as np

splits = load_splits(base.split_dir)
test_ds = KvasirSegDataset(base.data_root, splits["test"],
                           SegTransform(base.image_size, train=False))
img, msk = test_ds[0]

for cfg_ in up_configs:
    model = load_best(build_model(cfg_), cfg_.ckpt_path, get_device())
    pred, _ = predict_batch(model, img.unsqueeze(0), get_device())
    fig = zoom_on_border(denormalize(img), pred[0, 0].numpy(), box=(90, 90, 70, 70),
                         save_path=f"{base.fig_dir}/zoom_{cfg_.up_mode}.png")
    fig.suptitle(cfg_.up_mode)

## Câu hỏi cần trả lời

1. `bilinear` có ít artifact hơn `transpose` không? Có bằng chứng hình ảnh chưa?
2. Bỏ skip làm Dice tụt bao nhiêu? Tụt nhiều nhất ở loại ảnh nào — polyp nhỏ
   hay polyp có biên phức tạp?
3. `half skip` nằm ở đâu giữa `full` và `none`? Nếu nó gần `full` thì kết luận
   là gì về vai trò của các tầng nông?